# 07 — Validate the model: reconstructions + spectra + Skymap moments

Loads a trained U-Net and produces three figures and one headline number:

1. **`reconstructions.png`** — 4 random validation samples, side-by-side TRUE / MASKED INPUT / U-Net RECONSTRUCTION / |error| at one energy slice.
2. **`spectra.png`** — mean per-energy integrated PSD: true vs. U-Net inpainted vs. baseline inpainted vs. masked input.
3. **`moments_density.png`** — plasma density time series computed from the project's `Skymap` class, true vs. U-Net inpainted vs. baseline inpainted.
4. **median |Δn|/n** — median fractional density error vs. truth (the moments-level number, lower is better).

*Important:* validation here means the proper inpainting use case — keep the truth in visible bins, only fill the occluded region with the model's prediction. Running moments on the model's raw output everywhere would double-penalise it on bins it doesn't own.

In [ ]:
import sys, os, warnings, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd())=='notebooks' else os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'src'))
sys.path.insert(0, os.path.join(ROOT, 'MMS-FPI-Data-Gaps'))
import data_pipeline as dp
from model import build_unet3d
from baseline import energy_shell_mean_fill
from skymaps.skymap import Skymap

RUN = 'unet_full_3rounds'    # which trained run to inspect
BASE_F = 10                   # must match how this run was trained
IN_CHANNELS = 5               # 3 (temporal_window=1) + pitch_angle + logb
print('inspecting outputs/' + RUN)

## Load model + validation data


In [ ]:
mask = dp.synthetic_wedge_mask()
files = dp.find_dist_files(os.path.join(ROOT, 'MMS-FPI-Data-Gaps'))
_, val_files = dp.split_files(files, val_fraction=0.2, seed=0)
Xs, Ys = [], []
for f in val_files:
    fb = dp.read_dist_file(f, subsample=24, with_phi=True)
    X, Y = dp.build_inputs(fb, mask, use_pitch_angle=True, use_logb=True, temporal_window=1)
    Xs.append(X); Ys.append(Y)
X = np.concatenate(Xs, 0); Y = np.concatenate(Ys, 0)
print('val:', Y.shape, 'channels:', X.shape[-1])

m = build_unet3d(base_filters=BASE_F, input_shape=(32, 16, 32, IN_CHANNELS))
m.load_weights(os.path.join(ROOT, 'outputs', RUN, 'model_latest.h5'))
P = m.predict(X, verbose=0)

## 1. Reconstruction visualisation


In [ ]:
rng = np.random.default_rng(7)
idx_show = rng.choice(Y.shape[0], 4, replace=False)
e_bin = 16
fig, axes = plt.subplots(4, 4, figsize=(12, 11))
for row, i in enumerate(idx_show):
    true = Y[i, e_bin, :, :, 0]; masked_in = X[i, e_bin, :, :, 1]
    pred = P[i, e_bin, :, :, 0]; diff = np.abs(true - pred)
    vmax = max(true.max(), pred.max(), 1e-6)
    axes[row,0].imshow(true, vmin=0, vmax=vmax, origin='lower'); axes[row,0].set_title(f'sample {i}: TRUE')
    axes[row,1].imshow(masked_in, vmin=0, vmax=vmax, origin='lower'); axes[row,1].set_title('MASKED input')
    axes[row,2].imshow(pred, vmin=0, vmax=vmax, origin='lower'); axes[row,2].set_title('U-Net reconstruction')
    axes[row,3].imshow(diff, vmin=0, vmax=vmax*0.5, cmap='magma', origin='lower'); axes[row,3].set_title('|error|')
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f'U-Net inpainting -- middle energy slice (bin {e_bin})')
fig.tight_layout(); plt.show()

## 2. Per-energy spectrum (proper inpainting: truth in visible bins, prediction in occluded bins)


In [ ]:
y_cube = Y[..., 0]; p_cube = P[..., 0]
masked_cubes = dp.apply_mask(y_cube, mask)
b_cube = energy_shell_mean_fill(masked_cubes, mask)
inpaint = y_cube.copy(); inpaint[:, mask] = p_cube[:, mask]
b_in = y_cube.copy();    b_in[:, mask]    = b_cube[:, mask]
def spec(cube): return dp.to_physical_space(cube).sum(axis=(2,3))
for label, cube, sty in [('true', y_cube, 'k-'), ('U-Net inpaint', inpaint, 'C0-'),
                          ('baseline inpaint', b_in, 'C3--'), ('masked input', masked_cubes, 'C7:')]:
    plt.plot(spec(cube).mean(0), sty, label=label)
plt.yscale('log'); plt.xlabel('energy bin'); plt.ylabel('integrated PSD (mean over val)')
plt.title('Per-energy spectrum'); plt.legend(); plt.show()

## 3. Moments via the project's `Skymap` class

Note: the `Skymap` class in this repo allocates `self.skymap = np.empty((steps, 16, 32, nen))` but then iterates `self.skymap[t, energy_i]`, which only matches a `(steps, ENERGY, theta, phi)` layout. We override the allocation accordingly.

In [ ]:
N_MOM = min(80, y_cube.shape[0])           # moments integration is pure-Python, slow
sel = np.random.default_rng(3).choice(y_cube.shape[0], N_MOM, replace=False); sel.sort()
def to_sky(cube): return dp.to_physical_space(cube[sel])  # (N, 32, 16, 32)
moms = {}
for name, cube in [('true', y_cube), ('U-Net inpaint', inpaint), ('baseline inpaint', b_in)]:
    sk = Skymap(N_MOM, name=name)
    sk.skymap = np.zeros((N_MOM, 32, 16, 32), dtype=np.float64)
    sk.skymap[:] = to_sky(cube)
    print('computing', name, '...')
    moms[name] = sk.momsTS

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(moms['true'].density,             'k-',  lw=2.0, label='true')
ax.plot(moms['U-Net inpaint'].density,    'C0-', lw=1.4, label='U-Net inpaint')
ax.plot(moms['baseline inpaint'].density, 'C3--', lw=1.2, label='baseline inpaint')
ax.set_xlabel('validation sample (time order)'); ax.set_ylabel('density (Skymap moments)')
ax.set_title('Plasma density: reconstructed cubes vs. ground truth')
ax.legend(); plt.show()

d_true = np.asarray(moms['true'].density)
for name in ['U-Net inpaint', 'baseline inpaint']:
    d = np.asarray(moms[name].density)
    m = np.abs(d_true) > 1e-30
    print(f'median |Δn|/n  {name:18s} = {np.median(np.abs(d_true[m]-d[m])/np.abs(d_true[m])):.3f}')